In [1]:
# ✅ Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ✅ Step 2: Unzip Datasets
import zipfile, os

base_path = '/content/drive/MyDrive/DeepfakeDataset/'

zips = {
    'image': 'ImageDataset.zip',
    'video': 'SDFVD.zip'
}
dest_paths = {
    'image': '/content/image_data/ImageDataset',
    'video': '/content/video_data/ImageDataset'
}

for key, zipname in zips.items():
    with zipfile.ZipFile(base_path + zipname, 'r') as zip_ref:
        zip_ref.extractall(dest_paths[key])
    print(f"{zipname} extracted to {dest_paths[key]}")


ImageDataset.zip extracted to /content/image_data/ImageDataset
SDFVD.zip extracted to /content/video_data/ImageDataset


In [3]:
# ✅ Step 3: Install Dependencies
!pip install -q tensorflow opencv-python-headless

# ✅ Step 4: Import Libraries
import numpy as np
import cv2
import os
from PIL import Image, UnidentifiedImageError
import tensorflow as tf
from sklearn.utils import class_weight
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import (Input, Dense, GlobalAveragePooling2D, Dropout,
                                     BatchNormalization, RandomFlip, RandomRotation,
                                     RandomZoom)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

In [4]:
# ✅ Step 5: Clean Corrupted Images
def clean_dataset(directory, allowed_exts={'.jpg', '.jpeg', '.png', '.bmp', '.gif'}):
    bad_files = []
    for subdir, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(subdir, file)
            ext = os.path.splitext(file)[1].lower()

            if ext not in allowed_exts:
                bad_files.append(file_path)
                continue

            try:
                with Image.open(file_path) as img:
                    img.verify()
            except (UnidentifiedImageError, OSError):
                bad_files.append(file_path)
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

    print(f"Found {len(bad_files)} bad files.")
    for bf in bad_files:
        try:
            os.remove(bf)
            print(f"Removed: {bf}")
        except Exception as e:
            print(f"Failed to remove {bf}: {e}")

clean_dataset('/content/image_data')

Streaming output truncated to the last 5000 lines.
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_3986.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_4808.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_3108.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_6224.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_2588.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_10038.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_4580.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_7342.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_9561.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_5089.jpg
Removed: /content/image_data/ImageDataset/__MACOSX/ImageDataset/Real/._real_4527.jpg
Removed: /con

In [5]:
# ✅ Step 6: Load Dataset
image_dir = '/content/image_data/ImageDataset/ImageDataset'

train_ds = tf.keras.utils.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(128, 128),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(128, 128),
    batch_size=32
)

class_names = train_ds.class_names
print("Class names:", class_names)


Found 20383 files belonging to 2 classes.
Using 16307 files for training.
Found 20383 files belonging to 2 classes.
Using 4076 files for validation.
Class names: ['Fake', 'Real']


In [6]:
# ✅ Step 8: Normalize + Augment
from tensorflow.keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Sequential

data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1)
])

normalization_layer = Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(data_augmentation(x, training=True)), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [7]:
# ✅ Step 9: Prefetch for Performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [8]:
# ✅ Step 10: Calculate Class Weights (handle imbalance)
import numpy as np
from sklearn.utils import class_weight

labels = np.concatenate([y.numpy() for x, y in train_ds], axis=0)
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)


Class weights: {0: np.float64(0.9821127439171284), 1: np.float64(1.0185509056839475)}


In [9]:
print("Class names:", class_names)


Class names: ['Fake', 'Real']


In [10]:
# ✅ Step 11: Build Model (EfficientNetB0)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

input_img = Input(shape=(128, 128, 3))
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=input_img)
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=input_img, outputs=output)

from tensorflow.keras.metrics import AUC, Precision, Recall

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        AUC(name='auc', curve='ROC'),
        Precision(name='precision'),
        Recall(name='recall')
    ]
)

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 128, 128,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 128, 128,  │          7 │ rescaling_1[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 128, 128,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 129, 129,  │          0 │ rescaling_2[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 64, 64,    │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 64, 64,    │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 64, 64,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 64, 64,    │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 64, 64,    │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 64, 64,    │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 64, 64,    │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 64, 64,    │        512 │ block1a_se_excit

 Total params: 4,214,180 (16.08 MB)

 Trainable params: 4,171,901 (15.91 MB)

 Non-trainable params: 42,279 (165.16 KB)

In [12]:
# ✅ Step 12: Train Model with Callbacks
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('/content/drive/MyDrive/bestie_model.h5', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    class_weight=class_weights
)

Epoch 1/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.9055 - auc: 0.9682 - loss: 0.2295 - precision: 0.9050 - recall: 0.9050

510/510 ━━━━━━━━━━━━━━━━━━━━ 76s 148ms/step - accuracy: 0.9055 - auc: 0.9682 - loss: 0.2295 - precision: 0.9050 - recall: 0.9050 - val_accuracy: 0.5074 - val_auc: 0.5824 - val_loss: 1.0648 - val_precision: 0.5051 - val_recall: 0.9976 - learning_rate: 1.0000e-04
Epoch 2/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.9275 - auc: 0.9801 - loss: 0.1787 - precision: 0.9224 - recall: 0.9321

510/510 ━━━━━━━━━━━━━━━━━━━━ 83s 149ms/step - accuracy: 0.9274 - auc: 0.9801 - loss: 0.1787 - precision: 0.9224 - recall: 0.9321 - val_accuracy: 0.6320 - val_auc: 0.7147 - val_loss: 0.7555 - val_precision: 0.5856 - val_recall: 0.9165 - learning_rate: 1.0000e-04
Epoch 3/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.9340 - auc: 0.9832 - loss: 0.1644 - precision: 0.9302 - recall: 0.9373

510/510 ━━━━━━━━━━━━━━━━━━━━ 82s 150ms/step - accuracy: 0.9340 - auc: 0.9833 - loss: 0.1644 - precision: 0.9302 - recall: 0.9373 - val_accuracy: 0.6183 - val_auc: 0.6601 - val_loss: 0.7458 - val_precision: 0.6103 - val_recall: 0.6657 - learning_rate: 1.0000e-04
Epoch 4/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.9457 - auc: 0.9888 - loss: 0.1341 - precision: 0.9413 - recall: 0.9500

510/510 ━━━━━━━━━━━━━━━━━━━━ 86s 158ms/step - accuracy: 0.9457 - auc: 0.9888 - loss: 0.1341 - precision: 0.9413 - recall: 0.9500 - val_accuracy: 0.7252 - val_auc: 0.8086 - val_loss: 0.5498 - val_precision: 0.6979 - val_recall: 0.7994 - learning_rate: 1.0000e-04
Epoch 5/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 78s 152ms/step - accuracy: 0.9497 - auc: 0.9900 - loss: 0.1259 - precision: 0.9466 - recall: 0.9523 - val_accuracy: 0.5020 - val_auc: 0.5025 - val_loss: 1.0565 - val_precision: 0.5025 - val_recall: 0.9444 - learning_rate: 1.0000e-04
Epoch 6/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 74s 144ms/step - accuracy: 0.9574 - auc: 0.9917 - loss: 0.1130 - precision: 0.9557 - recall: 0.9587 - val_accuracy: 0.6084 - val_auc: 0.6310 - val_loss: 0.9505 - val_precision: 0.5713 - val_recall: 0.8853 - learning_rate: 1.0000e-04
Epoch 7/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.9624 - auc: 0.9931 - loss: 0.0998 - precision: 0.9593 - recall: 0.9651

510/510 ━━━━━━━━━━━━━━━━━━━━ 88s 156ms/step - accuracy: 0.9624 - auc: 0.9931 - loss: 0.0998 - precision: 0.9593 - recall: 0.9651 - val_accuracy: 0.7932 - val_auc: 0.8873 - val_loss: 0.4699 - val_precision: 0.7406 - val_recall: 0.9058 - learning_rate: 1.0000e-04
Epoch 8/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 79s 150ms/step - accuracy: 0.9621 - auc: 0.9933 - loss: 0.1025 - precision: 0.9594 - recall: 0.9643 - val_accuracy: 0.7475 - val_auc: 0.8198 - val_loss: 0.5981 - val_precision: 0.6977 - val_recall: 0.8785 - learning_rate: 1.0000e-04
Epoch 9/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 78s 152ms/step - accuracy: 0.9695 - auc: 0.9951 - loss: 0.0851 - precision: 0.9661 - recall: 0.9727 - val_accuracy: 0.5846 - val_auc: 0.7837 - val_loss: 1.3655 - val_precision: 0.8260 - val_recall: 0.2201 - learning_rate: 1.0000e-04
Epoch 10/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.9679 - auc: 0.9954 - loss: 0.0831 - precision: 0.9651 - recall: 0.9706
Epoch 10: ReduceLROnPlateau reducing learning rate t

510/510 ━━━━━━━━━━━━━━━━━━━━ 76s 148ms/step - accuracy: 0.9747 - auc: 0.9972 - loss: 0.0653 - precision: 0.9712 - recall: 0.9780 - val_accuracy: 0.8869 - val_auc: 0.9780 - val_loss: 0.3428 - val_precision: 0.9818 - val_recall: 0.7897 - learning_rate: 2.0000e-05
Epoch 12/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 84s 152ms/step - accuracy: 0.9751 - auc: 0.9970 - loss: 0.0655 - precision: 0.9726 - recall: 0.9776 - val_accuracy: 0.8143 - val_auc: 0.9494 - val_loss: 0.5677 - val_precision: 0.9641 - val_recall: 0.6550 - learning_rate: 2.0000e-05
Epoch 13/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 74s 145ms/step - accuracy: 0.9828 - auc: 0.9983 - loss: 0.0514 - precision: 0.9807 - recall: 0.9847 - val_accuracy: 0.8211 - val_auc: 0.9497 - val_loss: 0.5671 - val_precision: 0.9661 - val_recall: 0.6676 - learning_rate: 2.0000e-05
Epoch 14/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.9810 - auc: 0.9980 - loss: 0.0549 - precision: 0.9799 - recall: 0.9821
Epoch 14: ReduceLROnPlateau reducing learning rate

510/510 ━━━━━━━━━━━━━━━━━━━━ 76s 150ms/step - accuracy: 0.9824 - auc: 0.9985 - loss: 0.0481 - precision: 0.9804 - recall: 0.9844 - val_accuracy: 0.9404 - val_auc: 0.9900 - val_loss: 0.1674 - val_precision: 0.9803 - val_recall: 0.8995 - learning_rate: 4.0000e-06
Epoch 16/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 81s 149ms/step - accuracy: 0.9835 - auc: 0.9984 - loss: 0.0471 - precision: 0.9829 - recall: 0.9840 - val_accuracy: 0.9257 - val_auc: 0.9858 - val_loss: 0.2186 - val_precision: 0.9845 - val_recall: 0.8658 - learning_rate: 4.0000e-06
Epoch 17/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 78s 153ms/step - accuracy: 0.9819 - auc: 0.9977 - loss: 0.0543 - precision: 0.9836 - recall: 0.9800 - val_accuracy: 0.9347 - val_auc: 0.9878 - val_loss: 0.1914 - val_precision: 0.9811 - val_recall: 0.8873 - learning_rate: 4.0000e-06
Epoch 18/20
510/510 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.9855 - auc: 0.9984 - loss: 0.0441 - precision: 0.9824 - recall: 0.9885
Epoch 18: ReduceLROnPlateau reducing learning rate

In [16]:
model.save('/content/drive/MyDrive/bestie_model.keras')
